# Random World-Space Skeleton Animation

Loads the world-space transforms generated in `../motion_2/data`, selects a random clip, and renders its skeleton as an inline 3D animation.

In [ ]:
# Import visualization dependencies and locate the generated motion dataset.
# This cell sets up the necessary libraries for 3D visualization and animation,
# then locates the motion data directory by searching upward from the current working directory.
# The data directory contains world-space transforms generated by the motion preprocessing pipeline.
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation


def find_motion_data_dir():
    # Walk upward so the notebook works from nested working directories.
    for directory in (Path.cwd(), *Path.cwd().parents):
        candidate = directory / 'motion_2' / 'data'
        if (candidate / 'all_world_transforms.npy').is_file():
            return candidate
    raise FileNotFoundError(
        'Could not find motion_2/data/all_world_transforms.npy from the notebook working directory.'
    )


DATA_DIR = find_motion_data_dir()
print(f'Using motion data: {DATA_DIR}')

In [ ]:
# Load world transforms and build indexed skeleton connections for visualization.
# This cell loads the motion data files: world-space transforms (4x4 matrices for each joint),
# clip frame ranges (start/end frames for each motion clip), and skeleton structure metadata.
# It then constructs bone pairs as parent-child index tuples for rendering the skeleton.
import sys

sys.path.insert(0, str(DATA_DIR.parent))
all_world_transforms = np.load(DATA_DIR / 'all_world_transforms.npy', mmap_mode='r')

with open(DATA_DIR / 'clip_frame_ranges_world.pkl', 'rb') as file:
    clip_frame_ranges = pickle.load(file)

with open(DATA_DIR / 'skeleton_structure.pkl', 'rb') as file:
    skeleton_structure = pickle.load(file)

joint_names = list(skeleton_structure['joint_names'])
joint_hierarchy = skeleton_structure['joint_hierarchy']
joint_index = {name: index for index, name in enumerate(joint_names)}
bone_pairs = [
    (joint_index[parent_name], joint_index[joint_name])
    for joint_name, parent_name in joint_hierarchy.items()
    if parent_name is not None
]

if all_world_transforms.shape[1] != len(joint_names):
    raise ValueError('Transform joint count does not match skeleton metadata.')

print(f'World transforms: {all_world_transforms.shape}')
print(f'Available clips: {len(clip_frame_ranges)}')

In [ ]:
# Build one root-feature row for every pose in the world-transform dataset.
# This cell extracts root features (hip position and forward direction) for all poses.
# The forward direction is derived from the cross product of the hip-to-hip vector and world up.
# The resulting feature matrix contains [hips_x, hips_z, forward_x, forward_z] for each frame.
# This matrix is saved to disk for use in path prediction models.
hips_index = joint_index['Hips']
right_hip_index = joint_index['RightHip']
left_hip_index = joint_index['LeftHip']

all_hips_xz = np.asarray(all_world_transforms[:, hips_index, [0, 2], 3])
all_right_to_left_xz = (
    np.asarray(all_world_transforms[:, right_hip_index, [0, 2], 3])
    - np.asarray(all_world_transforms[:, left_hip_index, [0, 2], 3])
)
all_forward_xz = np.column_stack(
    (-all_right_to_left_xz[:, 1], all_right_to_left_xz[:, 0])
)
all_forward_lengths = np.linalg.norm(all_forward_xz, axis=1, keepdims=True)

if np.any(all_forward_lengths == 0):
    raise ValueError('Cannot derive a forward axis for poses where the hips overlap.')

all_forward_xz /= all_forward_lengths
root_feature_matrix = np.column_stack((all_hips_xz, all_forward_xz))

output_data_dir = Path.cwd() / 'data'
output_data_dir.mkdir(parents=True, exist_ok=True)
root_feature_matrix_path = output_data_dir / 'root_feature_matrix.npy'
np.save(root_feature_matrix_path, root_feature_matrix)

print('Root feature matrix shape:', root_feature_matrix.shape)
print('Columns: hips_x, hips_z, forward_x, forward_z')
print(f'Saved root features to: {root_feature_matrix_path}')
print(root_feature_matrix[:5])

In [ ]:
# Extract every source frame from one randomly selected clip before calculating features.
# This cell randomly selects a motion clip from the dataset and extracts its joint positions.
# It also downsamples the frames to a maximum of 600 frames for efficient rendering in the animation.
# The downsampled positions are used for the inline animation, while full positions are used for feature extraction.
rng = np.random.default_rng()
clip_name = rng.choice(list(clip_frame_ranges))
start_frame, end_frame = clip_frame_ranges[clip_name]
clip_positions = np.asarray(all_world_transforms[start_frame:end_frame, :, :3, 3])

# Downsample only the frames passed to the inline animation renderer.
max_display_frames = 600
frame_stride = max(1, int(np.ceil(len(clip_positions) / max_display_frames)))
display_frame_indices = np.arange(start_frame, end_frame, frame_stride)
display_positions = clip_positions[::frame_stride]

print(f'Random clip: {clip_name}')
print(f'Source frames: {len(clip_positions)}; displayed frames: {len(display_positions)}')
print(f'Frame stride: {frame_stride}')

In [ ]:
# Derive and plot the root trajectory and forward direction from hip positions.
# This cell calculates the root trajectory (hip positions over time) and forward direction
# for the selected clip. It then creates a 2D plot showing the path with forward direction arrows.
# The plot visualizes the character's movement through world space with orientation indicators.
hips_index = joint_index['Hips']
right_hip_index = joint_index['RightHip']
left_hip_index = joint_index['LeftHip']

hips_xz = clip_positions[:, hips_index, [0, 2]]
right_to_left_xz = (
    clip_positions[:, right_hip_index, [0, 2]]
    - clip_positions[:, left_hip_index, [0, 2]]
)
right_to_left_3d = np.column_stack(
    (right_to_left_xz[:, 0], np.zeros(len(right_to_left_xz)), right_to_left_xz[:, 1])
)
forward_3d = np.cross(np.array([0.0, 1.0, 0.0]), right_to_left_3d)
forward_lengths = np.linalg.norm(forward_3d, axis=1, keepdims=True)

if np.any(forward_lengths == 0):
    raise ValueError('Cannot derive a forward axis where the hips overlap.')

forward_xz = forward_3d[:, [0, 2]] / forward_lengths
clip_root_feature_matrix = np.column_stack((hips_xz, forward_xz))

print('Clip root feature matrix shape:', clip_root_feature_matrix.shape)
print('Columns: hips_x, hips_z, forward_x, forward_z')
print(clip_root_feature_matrix[:5])

figure, axis = plt.subplots(figsize=(9, 8))
axis.plot(hips_xz[:, 0], hips_xz[:, 1], color='#1d3557', linewidth=1.5, label='Hip trajectory')

arrow_stride = max(1, len(clip_root_feature_matrix) // 60)
arrow_indices = np.arange(0, len(clip_root_feature_matrix), arrow_stride)
path_extent = np.ptp(hips_xz, axis=0)
arrow_length = max(path_extent) * 0.05
axis.quiver(
    hips_xz[arrow_indices, 0], hips_xz[arrow_indices, 1],
    forward_xz[arrow_indices, 0] * arrow_length,
    forward_xz[arrow_indices, 1] * arrow_length,
    color='#d1495b', angles='xy', scale_units='xy', scale=1, width=0.006,
    headwidth=4, headlength=5, zorder=4, label='Forward axis',
)
axis.scatter(*hips_xz[0], color='#2a9d8f', s=60, zorder=3, label='Start')
axis.scatter(*hips_xz[-1], color='#e76f51', s=60, zorder=3, label='End')
axis.set_title(f'Root Trajectory and Forward Axis: {clip_name}')
axis.set_xlabel('World X')
axis.set_ylabel('World Z')
axis.set_aspect('equal')
axis.grid(alpha=0.25)
axis.legend()
figure.tight_layout()
plt.show()

In [ ]:
# Render the selected clip as an inline 3D skeleton animation.
# This cell creates a 3D matplotlib animation of the skeleton using the downsampled positions.
# It sets up the 3D plot with appropriate coordinate bounds and aspect ratio, then animates
# the joints (as scatter points) and bones (as line segments) frame by frame.
# The animation is rendered as JavaScript HTML for inline display in the notebook and saved as a video file.
coordinate_min = clip_positions.min(axis=(0, 1))
coordinate_max = clip_positions.max(axis=(0, 1))
span = np.maximum(coordinate_max - coordinate_min, 1e-6)
padding = max(span) * 0.08

figure = plt.figure(figsize=(7, 7))
axis = figure.add_subplot(111, projection='3d')
axis.set_xlim(coordinate_min[0] - padding, coordinate_max[0] + padding)
axis.set_ylim(coordinate_min[2] - padding, coordinate_max[2] + padding)
axis.set_zlim(coordinate_min[1] - padding, coordinate_max[1] + padding)
axis.set_box_aspect((span[0], span[2], span[1]))
axis.set_proj_type('ortho')
axis.set_xlabel('X')
axis.set_ylabel('Z')
axis.set_zlabel('Y')

initial_positions = display_positions[0]
joints = axis.scatter(
    initial_positions[:, 0], initial_positions[:, 2], initial_positions[:, 1],
    color='#d1495b', s=28, depthshade=False
)
bones = [axis.plot([], [], [], color='#1d3557', linewidth=2)[0] for _ in bone_pairs]


def update(frame_number):
    positions = display_positions[frame_number]
    joints._offsets3d = (positions[:, 0], positions[:, 2], positions[:, 1])

    for bone, (parent_index, child_index) in zip(bones, bone_pairs):
        bone.set_data(
            [positions[parent_index, 0], positions[child_index, 0]],
            [positions[parent_index, 2], positions[child_index, 2]],
        )
        bone.set_3d_properties([positions[parent_index, 1], positions[child_index, 1]])

    axis.set_title(
        f'{clip_name} | frame {display_frame_indices[frame_number] - start_frame + 1}'
    )
    return [joints, *bones]


animation = FuncAnimation(
    figure, update, frames=len(display_positions), interval=1000 / 60 * frame_stride, blit=False
)

# Save animation as video file
output_data_dir = Path.cwd() / 'data'
output_data_dir.mkdir(parents=True, exist_ok=True)
video_path = output_data_dir / f'{clip_name}_skeleton_animation.mp4'

try:
    animation.save(str(video_path), writer='ffmpeg', fps=30, dpi=100)
    print(f'Animation saved as video: {video_path}')
except Exception as e:
    print(f'Could not save as MP4 (ffmpeg may not be available): {e}')
    # Fallback to GIF if ffmpeg is not available
    try:
        gif_path = output_data_dir / f'{clip_name}_skeleton_animation.gif'
        animation.save(str(gif_path), writer='pillow', fps=30, dpi=100)
        print(f'Animation saved as GIF: {gif_path}')
    except Exception as e2:
        print(f'Could not save as GIF either: {e2}')

plt.close(figure)
display(HTML(animation.to_jshtml()))

In [ ]:
# Build and print the skeleton's parent-child hierarchy as an indented tree.
# This cell reconstructs the skeleton hierarchy from the joint data and prints it
# as a readable indented tree structure. This helps visualize the bone relationships
# and understand the skeleton's topology for analysis and debugging.
children_by_parent = {joint_name: [] for joint_name in joint_names}
root_joints = []

for joint_name in joint_names:
    parent_name = joint_hierarchy[joint_name]
    if parent_name is None:
        root_joints.append(joint_name)
    else:
        children_by_parent[parent_name].append(joint_name)


def print_hierarchy(joint_name, prefix=''):
    print(f'{prefix}{joint_name}')
    for child_name in children_by_parent[joint_name]:
        print_hierarchy(child_name, prefix + '  ')


print('Skeleton hierarchy:')
for root_joint in root_joints:
    print_hierarchy(root_joint)

In [ ]:
# Draw a labeled 2D reference diagram of the skeleton hierarchy.
# This cell creates a static 2D visualization of the skeleton structure with manually
# positioned joints. It draws lines between parent-child joints and labels each joint.
# This provides a reference diagram to understand the skeleton's topology and joint naming.
joint_positions = {
    'Hips': (0.0, 0.0),
    'Chest': (0.0, 1.1),
    'Chest2': (0.0, 2.0),
    'Chest3': (0.0, 2.9),
    'Chest4': (0.0, 3.8),
    'Neck': (0.0, 4.7),
    'Head': (0.0, 5.7),
    'RightCollar': (1.0, 4.0),
    'RightShoulder': (2.0, 3.9),
    'RightElbow': (3.3, 3.1),
    'RightWrist': (4.4, 2.2),
    'LeftCollar': (-1.0, 4.0),
    'LeftShoulder': (-2.0, 3.9),
    'LeftElbow': (-3.3, 3.1),
    'LeftWrist': (-4.4, 2.2),
    'RightHip': (0.8, -0.1),
    'RightKnee': (1.0, -2.0),
    'RightAnkle': (1.0, -3.9),
    'RightToe': (1.6, -4.5),
    'LeftHip': (-0.8, -0.1),
    'LeftKnee': (-1.0, -2.0),
    'LeftAnkle': (-1.0, -3.9),
    'LeftToe': (-1.6, -4.5),
}

missing_positions = set(joint_names) - set(joint_positions)
if missing_positions:
    raise ValueError(f'Missing display positions for: {sorted(missing_positions)}')

figure, axis = plt.subplots(figsize=(8, 10))
for joint_name, parent_name in joint_hierarchy.items():
    if parent_name is not None:
        parent_x, parent_y = joint_positions[parent_name]
        child_x, child_y = joint_positions[joint_name]
        axis.plot([parent_x, child_x], [parent_y, child_y], color='#1d3557', linewidth=3, zorder=1)

axis.scatter(
    [joint_positions[joint_name][0] for joint_name in joint_names],
    [joint_positions[joint_name][1] for joint_name in joint_names],
    color='#d1495b', s=95, zorder=2,
)

for joint_name in joint_names:
    joint_x, joint_y = joint_positions[joint_name]
    axis.annotate(joint_name, (joint_x, joint_y), xytext=(0, 8), textcoords='offset points',
                  ha='center', fontsize=8)

axis.set_title('Skeleton Hierarchy')
axis.set_aspect('equal')
axis.axis('off')
axis.set_xlim(-5.2, 5.2)
axis.set_ylim(-5.1, 6.3)
figure.tight_layout()
plt.show()

In [ ]:
# Display the shape of the root feature matrix for reference.
# This cell shows the dimensions of the saved root feature matrix,
# which contains [num_poses, 4] where the 4 columns are hips_x, hips_z, forward_x, forward_z.
root_feature_matrix.shape

In [ ]:
# Empty cell for additional analysis or scratch work.
# This cell is intentionally left blank and can be used for testing, debugging,
# or running additional analysis code as needed.